# NorthWind PLUS business agent — Relate by Neo4j + Neo4j Aura + Claude

Ask business questions about the **live** NorthWind PLUS simulation. The agent looks up proven Cypher templates in its
**relate** knowledge store (the ontology plus the query library), runs read queries on **Neo4j Aura**, can make one
governed change (the L1/L2/L3 PO approval bands) — and **learns**: every question is kept in relate as an
`Interaction`, and new queries you approve become templates that later questions find.

**Before you start**
1. Load the graph: in the Aura Query console, run `model/NorthWind_PLUS_Graph_Data_Model_Import_v2.3.2.cypher`
   from line 95 (*"// Let's clear all Nodes..."*) to line 1171. This **wipes the database** and builds NorthWind PLUS v2.3.2.
   Optionally also run the Recommendation Engine (lines 1271-1334).
   *(Already on v2.3.1 with data you want to keep? Run `model/NorthWind_PLUS_v2.3.2_Upgrade_from_v2.3.1.cypher` instead.)*
   The import rebuilds the classic Northwind database as a graph-native application and extends it into a complete,
   360-degree business cycle: customer orders, low-stock detection, purchase orders with three levels of approval,
   supplier RFQs, warehouse receiving, finance closure and order fulfillment. All of it runs natively on the graph from
   here on: no data ingestion, no ETL or data pipelines.
2. Run `model/NorthwindPlusGraphType_v2.3.2.cypher` in the same console, to apply the GRAPH TYPE schema. This adds a
   governance layer to the model: the database itself enforces the ontology.
3. Start the six Python loops (`model/NorthwindPlus_Stress_Test.py`, one process per `--loop`), so the data is moving.
   On Windows, run `run_loops.bat` from the `NorthWindPLUS/Neo4jRelate` folder on your own machine, with `.env` filled
   in: it opens six consoles, one per business process, so you can watch what happens in the background.
   Your graph-native system of record is now running on Neo4j Aura. Next, an AI agent answers questions about that live
   graph, using its own embedded Relate graph.
4. Fill in `.env` in this folder (see `.env.example`): `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.
5. Use a kernel where `pip install -r requirements.txt` has been run.

Run the cells from the top. Each question is one `ask(...)` call.

## 1. Setup

In [ ]:
import logging, os, sys, json, subprocess
from pathlib import Path

PROJECT = Path.cwd()                   # the notebook lives in the project folder
assert (PROJECT / "agent" / "agent.py").exists(), f"Open this notebook from the project folder (found {PROJECT})"
for sub in ("agent", "kb"):
    sys.path.insert(0, str(PROJECT / sub))
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

from IPython.display import Markdown, display
import agent as northwind_agent
from build_kb import build
from kb_query import KnowledgeBase
print("Project folder:", PROJECT)
import shutil
cli = os.environ.get("RELATE_BIN") or shutil.which("relate")
print("relate CLI:", cli or "not found -- the what-if (step 5) will copy data through the Neo4j driver instead of relate pull")

## 2. Knowledge store (relate)

`EMBEDDINGS = "none"` uses full-text + graph + past-questions retrieval. `"voyage-4-nano"` adds semantic search with
Voyage AI's open-weight model, run on this machine (no API key; ~700 MB download the first time; needs
`sentence-transformers` and `torch`). Changing it rebuilds the store; **what the agent has learned is kept**.

In [ ]:
EMBEDDINGS = "none"        # or "voyage-4-nano"
REBUILD = False            # rebuild after editing kb/semantics.json or the model files (learning is kept)

store = PROJECT / "kb" / "northwind_kb.rl"
current = "none"
if store.exists():
    k = KnowledgeBase(store); current = k.embedding_model; k.close()
if REBUILD or not store.exists() or current != EMBEDDINGS:
    print(json.dumps(build(store, embeddings=EMBEDDINGS), indent=2))
else:
    print(f"Using {store.name} (embeddings: {current})")

## 3. Sign in

`EMPLOYEE_ID` stands in for a login — the session decides who you are, the model never does.

| EmployeeID | Who | Can change approval bands? |
|---|---|---|
| `25` | Olivia Bennett, ProcurementPolicyAdmin | yes |
| `12` | Gloria Gaynor, Level1Approver | no (can still ask questions) |

Rerunning this cell closes the previous session first (relate allows one writing connection per store).

In [ ]:
EMPLOYEE_ID = "25"
MODEL = northwind_agent.DEFAULT_MODEL      # e.g. "claude-opus-5-5" for harder questions

if "session" in globals():
    try:
        session.close()
    except Exception:
        pass
session = northwind_agent.Session(EMPLOYEE_ID, MODEL)
print(f"Signed in as {session.user['name']} ({', '.join(session.user['roles']) or 'no roles'}) -- model {session.model}")
print(f"Knowledge store: ontology {session.kb.ontology_version}, embeddings {session.kb.embedding_model}")


def ask(question: str):
    """Ask the agent. Tool calls print as they happen; the answer renders as Markdown."""
    display(Markdown(session.ask(question)))


def new_conversation():
    """Forget the conversation so far (same sign-in)."""
    session.messages.clear()
    print("Conversation cleared.")

## 4. Ask questions

The conversation carries over between cells, so follow-ups like *"and for last week?"* work. When the agent writes a
new query that worked, it offers to save it as a template: a text box appears under the cell — `y` to keep it.

In [ ]:
ask("Which suppliers get their quotes rejected most often?")

In [ ]:
ask("How many POs are waiting for approval?")

Ask the same thing in other words: if you saved the template above, the search should now find it — through
the template itself and through the past question it answered.

In [ ]:
ask("What is the number and value of purchase orders still waiting to be vetted?")

### Governed change

The agent previews first, then asks you to confirm in a text box under the cell: `y` to apply, anything else to cancel.
Only EmployeeID 25 can apply it.

In [ ]:
ask("Raise L1 to 8000 and L2 to 20000 to speed up vetting.")

In [ ]:
ask("Who changed the approval limits, when and why?")

### Your questions

Some ideas to start from (also in the README's *Things to ask*). Remove the `#` in front of one line and run the
cell, or copy a line into a new cell and change it: another time window, a top 5 instead of a top 10, another level.
One question per cell keeps each answer next to its question.

In [ ]:
# ask("What were total revenue and gross profit in the last hour?")        # or: in the last 15 minutes
# ask("Which available products urgently need restocking to fulfill customer orders?")
# ask("Who are our top 10 customers by revenue, and by product quantity?")
# ask("Who does the most PO vetting?")
# ask("Where is the PO vetting friction concentrated?")
# ask("Which supplier has the most rejected RFQs?")
# ask("Which available products are ordered least?")
# ask("Who is our best salesperson?")
# ask("Which is our best sales region?")
# ask("Which employee gives our customers the biggest discounts?")
# ask("Which fulfillment clerk received the most supplier deliveries (POs)?")   # or: customer orders
# ask("How are the POs waiting for approval spread across the L1, L2 and L3 bands?")
# ask("Which products are at risk of running out?")

In [ ]:
# ask("")

## 5. What-if: rehearse a change before making it

The agent pulls the POs waiting for approval from Aura into a **scratch relate store**, then runs the simulation's own
vetting Cypher there twice: with today's limits and with the proposed ones, 5 runs each (the simulation picks POs and
decisions at random). It compares approvals per level, rejections, POs left stuck and the cycles needed to clear the
queue. Nothing is written to Aura; the scratch stores are deleted afterwards.

The copy comes from `relate pull` when the `relate` command-line tool is installed, otherwise from a fallback copier
through the Neo4j driver. The answer says which, and how old the snapshot is. Resubmitted POs aren't replayed yet: a
rejection ends a PO's path in the rehearsal.

> **If the what-if shows little or no difference between the two scenarios**, the queue is probably too short to
> rehearse on. With the rates in `run_loops.bat`, the PO Vetting loop (12 per minute) clears new POs almost as fast as
> PO Creation (1 per minute) makes them. To build a backlog:
> - **pause the PO Vetting loop** (Ctrl+C in its window) for 10-15 minutes, run the what-if, then restart it; or
> - **slow it down**: change its `--rate 12` to 2 or 3 in `run_loops.bat`;
> - optionally **raise the Customer Order rate** (e.g. `--rate 6` to 12): POs include stock promised to open orders,
>   so busier sales mean bigger POs, and more of them land in the L2 and L3 bands. Keep an eye on the fulfillment window.
>
> Also check how the waiting POs spread across the bands first, e.g. `ask("How are the POs waiting for approval spread
> across the L1, L2 and L3 bands?")`. If almost all of them are under the L1 limit, raising it changes nothing.
> And note what the rehearsal measures: the simulation gives every approval level the same speed, so higher limits
> show up as fewer hand-offs between levels and fewer cycles to clear the queue, not as one level holding others up.

In [ ]:
ask("What would happen to the approval queue if we raised L1 to 8000 and L2 to 20000? Run a what-if.")

If the rehearsal looks right, apply it: the `PolicyChange` justification will cite the what-if's ID.

In [ ]:
# ask("Apply those limits: L1 8000, L2 20000, L3 unchanged.")

In [ ]:
rows = session.kb._rows("""MATCH (i:Interaction)-[:RAN]->(w:WhatIf)
    RETURN w.id AS id, w.createdAt AS at, w.method AS method, w.queuePOs AS pos, w.currentLimits AS cur,
           w.proposedLimits AS prop, w.appliedAs AS applied, i.question AS q ORDER BY at DESC LIMIT 10""")
display(Markdown("**What-ifs run so far**\n\n| What-if | POs pulled | Current limits | Proposed | Pulled with | Applied as |\n|---|---|---|---|---|---|\n" +
    "\n".join(f"| {r['id']} | {r['pos']} | {' / '.join(f'{v:,.0f}' for v in r['cur'])} | {' / '.join(f'{v:,.0f}' for v in r['prop'])} | "
               f"{r['method'].split(' ')[0]} | {r['applied'] or '-'} |" for r in rows) if rows else "_No what-if yet._"))

## 6. What the agent learned (from relate)

Templates the agent created, how often templates get reused, and the question history — all read from the knowledge
store, the same graph the agent searches.

In [ ]:
kb = session.kb
learned = kb._rows("""
    MATCH (q:QueryTemplate {source:'agent'})
    OPTIONAL MATCH (q)-[:PART_OF]->(w:WorkflowStep)
    RETURN q.title AS title, q.createdBy AS by, q.question AS firstQuestion, coalesce(q.useCount, 0) AS reuses,
           q.status AS status, collect(DISTINCT w.name) AS workflows ORDER BY reuses DESC""")
display(Markdown("**Learned templates**\n\n| Title | First question | Created by | Reuses | Workflows |\n|---|---|---|---|---|\n" +
    "\n".join(f"| {r['title']} | {r['firstQuestion']} | {r['by']} | {r['reuses']} | {', '.join(r['workflows'])} |" for r in learned)
    if learned else "_No learned templates yet._"))

top = kb._rows("""MATCH (q:QueryTemplate) WHERE q.useCount > 0
                 RETURN q.title AS title, q.source AS source, q.useCount AS uses ORDER BY uses DESC LIMIT 10""")
display(Markdown("**Most used templates**\n\n| Template | Source | Uses |\n|---|---|---|\n" +
    "\n".join(f"| {r['title'][:90]} | {r['source']} | {r['uses']} |" for r in top) if top else "_No template used yet._"))

hist = kb._rows("""MATCH (i:Interaction) OPTIONAL MATCH (i)-[:USED|CREATED]->(q:QueryTemplate)
                  RETURN i.question AS q, i.employeeId AS who, i.modelCalls AS calls, i.inputTokens AS tin,
                         i.cacheReadTokens AS tcache, i.outputTokens AS tout, collect(DISTINCT q.title) AS templates,
                         i.newQueries AS newq, i.startedAt AS t ORDER BY t DESC LIMIT 15""")
display(Markdown("**Recent questions**\n\n| Question | By | Templates | New queries | Model calls | Tokens in (+cached) / out |\n|---|---|---|---|---|---|\n" +
    "\n".join(f"| {r['q']} | {r['who']} | {'<br>'.join(t[:60] for t in r['templates']) or '-'} | {r['newq'] or 0} | {r['calls']} | "
               f"{(r['tin'] or 0):,} (+{(r['tcache'] or 0):,}) / {(r['tout'] or 0):,} |" for r in hist)))

## 7. Done

In [ ]:
session.close()
print("Closed the Aura connection and the knowledge store. What the agent learned stays in kb/northwind_kb.rl.")